# Image-only cameras + depth for RealCAD: Depth Anything 3 only (VGGT already done)

Input : `realcad_recon_in.zip` (made locally by `data/prep_realcad_recon_inputs.py`):
`realcad_recon/ids.txt`, `crops.json`, `images/<id>/<view>.png` (part cut out by its SAM 3 mask, white background,
one shared square crop per part, 518 x 518). Only the photos are used: no CAD, no ground truth.

Output: `realcad_recon_da3.zip` with `da3/<id>.npz`, holding
`views` (view numbers), `extrinsics` (V,3,4 OpenCV world-to-camera), `intrinsics` (V,3,3, pixels of `hw`),
`depth` (V,H,W float16), `conf` (V,H,W float16), `hw` (processed H, W), plus `log.txt`.

Kaggle: Settings > Accelerator > GPU T4; Internet on; attach the zip as a dataset; (optional) secret `HF_TOKEN`.
Re-running skips parts already done.
About 1-3 s per part per model on a T4.

In [ ]:
import os, sys, glob, json, time, zipfile, subprocess
from pathlib import Path
import numpy as np, torch
ON_KAGGLE = Path('/kaggle').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

token = os.environ.get('HF_TOKEN')
try:
    if not token and ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    token = None
if token:
    from huggingface_hub import login
    login(token=token)

def find_root():
    hits = glob.glob('/kaggle/input/**/crops.json', recursive=True) + glob.glob(str(WORK / '**/crops.json'), recursive=True)
    return Path(hits[0]).parent if hits else None
ROOT = find_root()
if ROOT is None:
    zips = glob.glob('/kaggle/input/**/realcad_recon_in.zip', recursive=True) + glob.glob(str(WORK / 'realcad_recon_in.zip'))
    if not zips and not ON_KAGGLE:
        from google.colab import files
        zips = [str(WORK / n) for n in files.upload()]
    zipfile.ZipFile(zips[0]).extractall(WORK)
    ROOT = find_root()
IDS = [l.strip() for l in (ROOT / 'ids.txt').read_text().split() if l.strip()]
CROPS = json.loads((ROOT / 'crops.json').read_text())
OUT = WORK / 'realcad_recon_da3'; OUT.mkdir(exist_ok=True)
LOG = open(OUT / 'log.txt', 'a')
def log(s):
    print(s, flush=True); LOG.write(s + '\n'); LOG.flush()
def view_paths(pid):
    return [str(ROOT / 'images' / pid / f'{k}.png') for k in CROPS[pid]['views']]
print(ROOT, len(IDS), 'parts')

## Depth Anything 3 (DA3-GIANT-1.1, falls back to DA3-LARGE-1.1)

In [ ]:
if not Path('/tmp/da3').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/ByteDance-Seed/Depth-Anything-3.git', '/tmp/da3'], check=True)
if True:
    # keep Kaggle's torch: install the package without deps, then only the deps that are missing
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--ignore-requires-python', '-e', '/tmp/da3'], check=True)
    import tomllib, re
    deps = tomllib.loads(Path('/tmp/da3/pyproject.toml').read_text()).get('project', {}).get('dependencies', [])
    skip = ('torch', 'torchvision', 'xformers', 'numpy')
    need = [d for d in deps if re.split(r'[<>=!~\[ ;]', d)[0].strip().lower() not in skip]
    for d in need:   # one at a time: a single failing package must not block the others
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--ignore-requires-python', d],
                           capture_output=True, text=True)
        print('ok    ' if r.returncode == 0 else 'FAILED', d)
sys.path.insert(0, '/tmp/da3/src')
# install whatever DA3 still imports but is missing (pyproject does not list everything, e.g. addict)
PIP = {'cv2': 'opencv-python', 'PIL': 'pillow', 'yaml': 'pyyaml', 'sklearn': 'scikit-learn', 'skimage': 'scikit-image'}
for attempt in range(25):
    for m in [m for m in sys.modules if m.startswith('depth_anything_3')]:
        del sys.modules[m]
    try:
        from depth_anything_3.api import DepthAnything3
        print('DA3 import OK'); break
    except ModuleNotFoundError as e:
        mod = (e.name or '').split('.')[0]
        if mod in ('xformers', 'torch', 'torchvision'):
            print('STOP: DA3 needs', mod); break
        pkg = PIP.get(mod, mod); print('installing', pkg)
        r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--ignore-requires-python', pkg],
                           capture_output=True, text=True)
        if r.returncode:
            print('pip failed for', pkg, r.stderr[-400:]); break

In [ ]:
def load_da3(name):
    from depth_anything_3.api import DepthAnything3   # a failure here shows the real missing module
    return DepthAnything3.from_pretrained(name).to('cuda').eval()

def run_da3():
    od = OUT / 'da3'; od.mkdir(exist_ok=True)
    model, name = None, None
    for cand in ('depth-anything/DA3-GIANT-1.1', 'depth-anything/DA3-LARGE-1.1'):
        try:
            model, name = load_da3(cand), cand
            break
        except Exception as e:
            log(f'da3 load {cand} failed: {type(e).__name__}: {e}')
    if model is None:
        raise RuntimeError('no DA3 checkpoint loaded')
    log(f'da3 using {name}')
    t0, n = time.time(), 0
    for pid in IDS:
        if (od / f'{pid}.npz').exists():
            continue
        try:
            with torch.no_grad():
                p = model.inference(view_paths(pid), process_res=504)
            conf = p.conf if getattr(p, 'conf', None) is not None else np.ones_like(p.depth)
            np.savez_compressed(od / f'{pid}.npz', views=np.array(CROPS[pid]['views']),
                extrinsics=np.asarray(p.extrinsics)[:, :3, :4], intrinsics=np.asarray(p.intrinsics),
                depth=np.asarray(p.depth).astype(np.float16), conf=np.asarray(conf).astype(np.float16),
                hw=np.array(np.asarray(p.depth).shape[-2:]), model=name)
            n += 1
        except torch.cuda.OutOfMemoryError as e:
            log(f'da3 {pid} OOM'); torch.cuda.empty_cache()
        except Exception as e:
            log(f'da3 {pid} FAILED {type(e).__name__}: {e}')
    log(f'da3: {n} parts in {time.time() - t0:.0f}s')
    del model; torch.cuda.empty_cache()

try:
    run_da3()
except Exception as e:
    log(f'da3 cell FAILED {type(e).__name__}: {e}')

In [ ]:
# package + download link
import shutil
for m in ('da3',):
    print(m, len(list((OUT / m).glob('*.npz'))) if (OUT / m).exists() else 0, 'parts')
shutil.make_archive(str(OUT), 'zip', OUT)
z = OUT.with_suffix('.zip'); print(z, round(z.stat().st_size / 1e6, 1), 'MB')
if ON_KAGGLE:
    from IPython.display import FileLink, display
    os.chdir(WORK); display(FileLink(z.name))
else:
    from google.colab import files
    files.download(str(z))